# Taller: Regresión Logística
Configuración inicial e importación de módulos.

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# Agregar carpeta src al path para importar módulos
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from modelo import sigmoid, proba, riesgo_logistico
from optimizacion import ajustar_modelo
from simulacion import generar_datos

# Datos base de los Ejercicios 1 y 2
x_base = np.array([-2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2])
y1_base = np.array([0, 0, 0, 1, 0, 1, 1, 1, 1])
yA_base = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1])

ImportError: attempted relative import with no known parent package

## Ejercicio 1: Superficie del riesgo logístico

In [ ]:
# (a) Evaluar sobre malla
theta0_vals = np.linspace(-4, 4, 100)
theta1_vals = np.linspace(-1, 6, 100)
T0, T1 = np.meshgrid(theta0_vals, theta1_vals)
R = np.zeros_like(T0)

for i in range(T0.shape[0]):
    for j in range(T0.shape[1]):
        R[i, j] = riesgo_logistico([T0[i, j], T1[i, j]], x_base, y1_base)

# (c) Minimizar desde (0,0)
theta_hat, R_min, res = ajustar_modelo(x_base, y1_base, theta0=(0.0, 0.0))
print(f"Theta_hat: {theta_hat}")
print(f"R_S(Theta_hat): {R_min}")

# (b) Gráficas 3D y Curvas de nivel
fig = plt.figure(figsize=(14, 6))

# Superficie 3D
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(T0, T1, R, cmap=cm.viridis, alpha=0.8)
ax1.scatter([theta_hat[0]], [theta_hat[1]], [R_min], color='red', s=100, label='Minimizador', zorder=5)
ax1.set_xlabel(r'$\theta_0$')
ax1.set_ylabel(r'$\theta_1$')
ax1.set_zlabel(r'$\hat{R}_S(\theta)$')
ax1.set_title('Superficie de Riesgo 3D')
ax1.legend()

# Curvas de nivel
ax2 = fig.add_subplot(122)
cp = ax2.contourf(T0, T1, R, levels=30, cmap=cm.viridis)
fig.colorbar(cp, ax=ax2)
ax2.scatter([theta_hat[0]], [theta_hat[1]], color='red', s=100, marker='x', label='Minimizador')
ax2.set_xlabel(r'$\theta_0$')
ax2.set_ylabel(r'$\theta_1$')
ax2.set_title('Curvas de Nivel')
ax2.legend()

plt.tight_layout()
plt.savefig('../resultados/ejercicio1_superficie_y_contorno.png', dpi=300, bbox_inches='tight')
plt.show()

# (d) Verificación numérica
R_shift_0 = riesgo_logistico(theta_hat + np.array([0.1, 0]), x_base, y1_base)
R_shift_1 = riesgo_logistico(theta_hat + np.array([0, 0.1]), x_base, y1_base)
print(f"\nVerificación (d):")
print(f"R_S(theta_hat) < R_S(theta_hat + (0.1, 0)): {R_min} < {R_shift_0} -> {R_min < R_shift_0}")
print(f"R_S(theta_hat) < R_S(theta_hat + (0, 0.1)): {R_min} < {R_shift_1} -> {R_min < R_shift_1}")

## Ejercicio 2: Datos separables y no separables

In [ ]:
# (a) y (b) Ajuste y Tabla
theta_A, R_A, _ = ajustar_modelo(x_base, yA_base)
theta_B, R_B, _ = ajustar_modelo(x_base, y1_base)

print("Conjunto | ||theta_hat||_2 | R_S(theta_hat)")
print(f"   A     | {np.linalg.norm(theta_A):.6f}  | {R_A:.6f}")
print(f"   B     | {np.linalg.norm(theta_B):.6f}  | {R_B:.6f}")

# (c) Gráficas de datos y curva logística
x_plot = np.linspace(-2.5, 2.5, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_data, theta, title in zip([ax1, ax2], [yA_base, y1_base], [theta_A, theta_B], ['Conjunto A (Separable)', 'Conjunto B (No Separable)']):
    ax.scatter(x_base, y_data, color='blue', zorder=5, label='Datos')
    ax.plot(x_plot, proba(x_plot, theta), color='red', label='Curva ajustada')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('../resultados/ejercicio2_ajustes.png', dpi=300, bbox_inches='tight')
plt.show()

# (d) Separable con maxiter
theta_A_10, R_A_10, _ = ajustar_modelo(x_base, yA_base, maxiter=10)
theta_A_100, R_A_100, _ = ajustar_modelo(x_base, yA_base, maxiter=100)

print("\nComparación Conjunto A (Separable):")
print(f"maxiter=10  -> ||theta||: {np.linalg.norm(theta_A_10):.4f}, Riesgo: {R_A_10:.6f}")
print(f"maxiter=100 -> ||theta||: {np.linalg.norm(theta_A_100):.4f}, Riesgo: {R_A_100:.6f}")
print("\nConclusión: La afirmación compatible es la II. La norma aumenta mientras el riesgo disminuye hacia cero.")

## Ejercicio 3: Recuperación de parámetros

In [ ]:
theta_star = np.array([-1.0, 2.0])
n_values = [50, 200, 1000]
results_ex3 = []

print("  n  | theta_0_hat | theta_1_hat |   e_n   ")
print("-------------------------------------------")
for n in n_values:
    X, Y = generar_datos(n)
    theta_hat_n, _, _ = ajustar_modelo(X, Y)
    e_n = np.linalg.norm(theta_hat_n - theta_star)
    results_ex3.append((n, theta_hat_n[0], theta_hat_n[1], e_n))
    print(f" {n:3} | {theta_hat_n[0]:11.4f} | {theta_hat_n[1]:11.4f} | {e_n:.5f}")

e_50 = results_ex3[0][3]
e_1000 = results_ex3[2][3]
print(f"\n¿e_1000 < e_50? {e_1000} < {e_50} -> {e_1000 < e_50}")

## Ejercicio 4: Interpretación de los parámetros

**(a)** Demostración de que $p_\theta(x) = 1/2 \iff x = -\theta_0/\theta_1$:
$$p_\theta(x) = \sigma(\theta_0 + \theta_1 x) = \frac{1}{1 + e^{-(\theta_0 + \theta_1 x)}}$$
Igualando a $1/2$:
$$\frac{1}{1 + e^{-(\theta_0 + \theta_1 x)}} = \frac{1}{2} \implies 1 + e^{-(\theta_0 + \theta_1 x)} = 2 \implies e^{-(\theta_0 + \theta_1 x)} = 1$$
Aplicando logaritmo natural:
$$-(\theta_0 + \theta_1 x) = 0 \implies \theta_0 + \theta_1 x = 0 \implies x = -\frac{\theta_0}{\theta_1}$$

In [ ]:
x_eval = np.linspace(-4, 4, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# (b) Fijar theta_1 = 1
theta1_fix = 1
for t0 in [-2, 0, 2]:
    p = proba(x_eval, [t0, theta1_fix])
    ax1.plot(x_eval, p, label=rf'$\theta_0={t0}$')
    x_star = -t0 / theta1_fix
    ax1.scatter([x_star], [0.5], color='black', zorder=5)
    print(f"theta_0={t0}: x* = {x_star}, p(x*) = {proba(np.array([x_star]), [t0, theta1_fix])[0]:.4f}")
ax1.set_title(r'$\theta_1 = 1$ fijo, variando $\theta_0$')
ax1.set_xlabel('x'); ax1.set_ylabel('p(x)'); ax1.legend(); ax1.grid(True, alpha=0.5)

# (c) Fijar theta_0 = 0
theta0_fix = 0
for t1 in [1, 2, 5]:
    p = proba(x_eval, [theta0_fix, t1])
    ax2.plot(x_eval, p, label=rf'$\theta_1={t1}$')
    diff = proba(np.array([1.0]), [theta0_fix, t1])[0] - proba(np.array([-1.0]), [theta0_fix, t1])[0]
    print(f"theta_1={t1}: p(1) - p(-1) = {diff:.4f}")
ax2.set_title(r'$\theta_0 = 0$ fijo, variando $\theta_1$')
ax2.set_xlabel('x'); ax2.set_ylabel('p(x)'); ax2.legend(); ax2.grid(True, alpha=0.5)

plt.tight_layout()
plt.savefig('../resultados/ejercicio4_interpretacion.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nConclusión (c): El modelo con theta_1=5 presenta la transición más abrupta (diferencia cercana a 1).")

## Ejercicio 5: Verosimilitud y estabilidad numérica

In [ ]:
X_5000, Y_5000 = generar_datos(5000)
p_star = proba(X_5000, theta_star)
p_star_clip = np.clip(p_star, 1e-15, 1 - 1e-15)

# (a) Cálculo directo
L_directa = np.prod((p_star ** Y_5000) * ((1 - p_star) ** (1 - Y_5000)))
log_L = np.sum(Y_5000 * np.log(p_star_clip) + (1 - Y_5000) * np.log(1 - p_star_clip))

print(f"L(theta_star) directa: {L_directa}")
print(f"log L(theta_star): {log_L}")

# (b) Verificación
print(f"\n¿L es 0.0? {L_directa == 0.0}")
print(f"¿log L es finito? {np.isfinite(log_L)}")

# (c) Riesgo y Delta
R_S_star = riesgo_logistico(theta_star, X_5000, Y_5000)
Delta = np.abs(-log_L - 5000 * R_S_star)
print(f"\nR_S(theta_star): {R_S_star}")
print(f"Delta = | -log L - n * R_S | : {Delta}")
print(f"¿Delta < 1e-8? {Delta < 1e-8}")